# معالجة البيانات وإعدادها باستخدام Tidyverse

خط معالجة بيانات عابر للغات البرمجة: تنزل Python البيانات ← تعالجها R باستخدام dplyr ← تعرض Python النتائج بصريًا.

يوضح هذا المصنف **SharedVFS** — نظام الملفات المشترك الذي يتيح لـ Python وR تبادل الملفات.

## 1. Python: تنزيل مجموعة البيانات

In [ ]:
import micropip
await micropip.install('pandas')
import pandas as pd, pyodide.http, os

url = "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv"
resp = await pyodide.http.pyfetch(url)
text = await resp.string()

os.makedirs("/shared/data", exist_ok=True)
with open("/shared/data/gapminder.csv", "w") as f:
    f.write(text)

df = pd.read_csv("/shared/data/gapminder.csv")
print(f"تم التنزيل: {df.shape[0]} صفوف، {df.shape[1]} أعمدة")
df.head()

## 2. R: تثبيت dplyr وtidyr وقراءة البيانات المشتركة

In [ ]:
install.packages(c("dplyr", "tidyr"))
library(dplyr)

gap <- read.csv("/shared/data/gapminder.csv")
cat("تمت القراءة من SharedVFS:", nrow(gap), "صفوف\n")
glimpse(gap)

## 3. dplyr: التلخيص حسب القارة (2007)

In [ ]:
gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    countries = n(),
    mean_life = round(mean(lifeExp), 1),
    median_gdp = round(median(gdpPercap), 0),
    total_pop = sum(as.numeric(pop))
  ) %>%
  arrange(desc(mean_life))

## 4. dplyr: أعلى معدلات الارتفاع في متوسط العمر المتوقع

In [ ]:
gains <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  select(country, continent, year, lifeExp) %>%
  tidyr::pivot_wider(names_from = year, values_from = lifeExp,
                     names_prefix = "y") %>%
  mutate(gain = y2007 - y1952) %>%
  arrange(desc(gain)) %>%
  head(10)
gains

## 5. dplyr: النمو السكاني حسب القارة

In [ ]:
pop_growth <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  group_by(continent, year) %>%
  summarize(total_pop = sum(as.numeric(pop)), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = year, values_from = total_pop,
                     names_prefix = "pop_") %>%
  mutate(growth_pct = round((pop_2007 / pop_1952 - 1) * 100, 1)) %>%
  arrange(desc(growth_pct))
pop_growth

## 6. R: كتابة النتائج إلى SharedVFS

In [ ]:
# كتابة ملخص القارات لتقوم Python بعرضه بصريًا
summary_2007 <- gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    mean_life = round(mean(lifeExp), 1),
    mean_gdp = round(mean(gdpPercap), 0),
    .groups = "drop"
  )
write.csv(summary_2007, "/shared/data/r_summary.csv", row.names = FALSE)
cat("تمت كتابة /shared/data/r_summary.csv\n")
summary_2007

## 7. Python: العرض المرئي لنتائج R

In [ ]:
import micropip
await micropip.install('plotly')
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

r_summary = pd.read_csv("/shared/data/r_summary.csv")
print("تمت القراءة من SharedVFS (مكتوبة بواسطة R):")
print(r_summary.to_string(index=False))

fig = px.bar(r_summary, x="continent", y="mean_life",
             title="متوسط العمر المتوقع حسب القارة (2007) — من R إلى Python",
             labels={"mean_life": "متوسط العمر المتوقع (بالسنوات)", "continent": "القارة"},
             color="continent")
fig.update_layout(template='plotly_dark', showlegend=False)
show_plotly(fig)

In [ ]:
fig = px.scatter(r_summary, x="mean_gdp", y="mean_life",
                 text="continent", size=[40]*len(r_summary),
                 title="الناتج المحلي الإجمالي مقابل متوسط العمر المتوقع حسب القارة (ملخص R → رسم Python)",
                 labels={"mean_gdp": "متوسط نصيب الفرد من الناتج المحلي الإجمالي", "mean_life": "متوسط العمر المتوقع"})
fig.update_traces(textposition="top center")
fig.update_layout(template='plotly_dark')
fig.update_yaxes(range=[r_summary['mean_life'].min() - 2, r_summary['mean_life'].max() + 6])
show_plotly(fig)

## أهم النقاط

- قامت **Python** بتنزيل بيانات CSV إلى `/shared/data/`
- قرأتها **R** عبر SharedVFS وعالجتها باستخدام خطوط أنابيب dplyr
- كتبت **R** الملخص مرة أخرى إلى `/shared/data/r_summary.csv`
- قرأت **Python** مخرجات R وأنشأت رسومات بيانية تفاعلية باستخدام Plotly

تتم مشاركة جميع الملفات عبر SharedVFS — دون الحاجة إلى استيراد أو تصدير يدوي.